# Scorelens: Dart-Modell feintunen (Kaggle / Colab, Gratis-GPU)

1. Datensatz vorher mit `tools/finetune/finetune_hf.sh DATEN --space` oder von Hand als privates HF-Dataset hochladen (enthält `train_hf.py`).
2. **Kaggle:** Settings → Accelerator *GPU T4 x2*, Secrets → `HF_TOKEN` hinzufügen. **Colab:** Laufzeit → T4, Schlüssel-Symbol links → Secret `HF_TOKEN`.
3. Unten `DATASET_REPO` / `OUTPUT_REPO` eintragen und alle Zellen ausführen. Auf Kaggle *Save & Run All* wählen, dann läuft es bis 12 h ohne offenes Fenster.


In [ ]:
%pip install -q -U ultralytics huggingface_hub
import os
os.environ['DATASET_REPO'] = 'DEIN-USER/scorelens-darts-ft1'   # <- anpassen
os.environ['OUTPUT_REPO']  = 'DEIN-USER/scorelens-dart-model-ft1'
os.environ['EPOCHS'] = '40'; os.environ['FREEZE'] = '10'; os.environ['BATCH'] = '16'; os.environ['RESUME'] = '1'
os.environ['WORK_DIR'] = '/kaggle/working/ft' if os.path.isdir('/kaggle/working') else '/content/ft'


In [ ]:
# HF-Token aus den Secrets (Kaggle oder Colab), sonst interaktiv
tok = None
try:
    from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    try:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN')
    except Exception:
        pass
if tok: os.environ['HF_TOKEN'] = tok
else:
    from huggingface_hub import login; login()


In [ ]:
from huggingface_hub import hf_hub_download
script = hf_hub_download(os.environ['DATASET_REPO'], 'train_hf.py', repo_type='dataset')
exec(open(script).read())   # trainiert, bewertet, lädt best.pt + metrics.json ins OUTPUT_REPO


Danach auf dem PC: `tools/finetune/finetune_hf.sh --fetch --name ft1` holt `best.pt`, zeigt die Metriken und exportiert das TFLite in die App.
